#COMBINA SEQUENCES + QUERYS -- ALINEAMIENTO MULTIPLE CLUSTAL OMEGA

In [ ]:
import os
import subprocess
from Bio import SeqIO

# === RUTAS ===
ruta_queries = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\data\Identificacion_patogeno\identifiacion_patogeno_individual_sequences"
ruta_hits_base = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\data\Identificacion_patogeno\Blast\Blastn_02_07_25\Top_Blastn_alignments"
ruta_output = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple"

# Crear carpeta de salida si no existe
os.makedirs(ruta_output, exist_ok=True)

# Procesar todos los archivos sequence_X.fasta
for archivo in os.listdir(ruta_queries):
    if archivo.startswith("sequence_") and archivo.endswith(".fasta"):
        numero = archivo.replace("sequence_", "").replace(".fasta", "")
        query_id = f"Query_{numero}"

        path_query = os.path.join(ruta_queries, archivo)
        path_hits = os.path.join(ruta_hits_base, query_id)

        if not os.path.exists(path_hits):
            print(f"No se encontró carpeta para {query_id}, saltando...")
            continue

        # Crear archivo combinado (query + top hits)
        archivo_combinado = os.path.join(ruta_output, f"{query_id}_combined.fasta")
        with open(archivo_combinado, "w") as salida:
            # Escribir la secuencia original
            for record in SeqIO.parse(path_query, "fasta"):
                SeqIO.write(record, salida, "fasta")
            # Escribir los 5 top hits
            for archivo_hit in os.listdir(path_hits):
                if archivo_hit.endswith(".fasta") or archivo_hit.endswith(".fa"):
                    path_hit_fasta = os.path.join(path_hits, archivo_hit)
                    for record in SeqIO.parse(path_hit_fasta, "fasta"):
                        SeqIO.write(record, salida, "fasta")

        # Archivo alineado
        archivo_alineado = os.path.join(ruta_output, f"{query_id}_alignment.aln")

        # Ejecutar Clustal Omega
        comando = [
            "clustalo",
            "-i", archivo_combinado,
            "-o", archivo_alineado,
            "--outfmt=clustal",
            "--force"
        ]

        try:
            subprocess.run(comando, check=True)
            print(f"✅ Alineamiento completo: {archivo_alineado}")
        except subprocess.CalledProcessError as e:
            print(f"❌ Error al alinear {query_id}: {e}")


✅ Alineamiento completo: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple\Query_1_alignment.aln
